<a href="https://colab.research.google.com/github/missstechie/Lightweight-Semantic-SDG-Classifier-MVP-/blob/main/Lightweight%20Semantic%20SDG%20Classifier%20(Low-Compute%20MVP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Lightweight Semantic SDG Classifier (MVP)

This project demonstrates a low-compute approach to mapping project descriptions to UN Sustainable Development Goals (SDGs) using semantic similarity.

Instead of training a heavy ML model, we use sentence embeddings to match input text with SDG descriptions.

###  Key Features
- Uses transformer-based embeddings
- Works without labeled training data
- Fast and cost-efficient
- Returns Top-3 SDG predictions

###  Use Case
Useful for tools like SDG tagging systems, open-source project classification, and policy analysis.

---

In [ ]:
!pip install -q sentence-transformers pandas scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
sdgs = {
    "SDG 1": "No poverty: ending poverty, financial inclusion, access to resources",
    "SDG 2": "Zero hunger: food security, agriculture, nutrition",
    "SDG 3": "Good health and well-being: healthcare, disease prevention, medical access",
    "SDG 4": "Quality education: schools, learning, digital education",
    "SDG 5": "Gender equality: women empowerment, equal rights",
    "SDG 6": "Clean water and sanitation: drinking water, hygiene",
    "SDG 7": "Clean energy: renewable energy, solar, electricity",
    "SDG 8": "Economic growth: jobs, employment, financial growth",
    "SDG 9": "Innovation and infrastructure: technology, industry",
    "SDG 10": "Reduced inequalities: social inclusion",
    "SDG 11": "Sustainable cities: urban development",
    "SDG 12": "Responsible consumption: sustainability",
    "SDG 13": "Climate action: environment, global warming",
    "SDG 14": "Life below water: oceans, marine life",
    "SDG 15": "Life on land: forests, biodiversity",
    "SDG 16": "Peace and justice: governance, institutions",
    "SDG 17": "Partnerships: collaboration, global cooperation"
}

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

def _predict_sdg_with_details(text):
    text_embedding = model.encode(text)
    similarities = cosine_similarity([text_embedding], sdg_embeddings)[0]

    sdg_preds = []
    for i, sim in enumerate(similarities):
        sdg_code = list(sdgs.keys())[i]
        sdg_description = sdgs[sdg_code]
        sdg_preds.append((sdg_code, sdg_description, sim))

    # Sort by similarity in descending order and return top 3
    return sorted(sdg_preds, key=lambda x: x[2], reverse=True)[:3]

def predict_sdg(text):
    preds_with_details = _predict_sdg_with_details(text)
    if preds_with_details:
        top_pred = preds_with_details[0]
        return (top_pred[0], top_pred[1])
    return (None, None)

def predict_sdg_top3(text):
    preds_with_details = _predict_sdg_with_details(text)
    return [p[0] for p in preds_with_details]

def evaluate(df_to_evaluate):
    correct = 0
    total_predictions = len(df_to_evaluate)

    print(f"Evaluating {total_predictions} projects...")
    for _, row in df_to_evaluate.iterrows():
        pred_code, _ = predict_sdg(row["project"])

        if pred_code == row["actual_sdg"]:
            correct += 1

    accuracy = correct / total_predictions
    print(f"Accuracy: {accuracy:.4f} ({correct}/{total_predictions})\n")

In [ ]:
print(predict_sdg_top3("A mobile app that improves rural healthcare and education access"))

In [ ]:
test_cases = [
    "A mobile app for maternal healthcare in rural areas",
    "AI platform for climate change monitoring",
    "Online education system for underprivileged students",
    "Solar panels for villages without electricity",
    "Microfinance platform for small businesses"
]

for t in test_cases:
    print("Input:", t)
    print("Top 3 SDGs:", predict_sdg_top3(t))
    print()

In [ ]:
data = {
    "project": [
        "Mobile health app for villages",
        "Online education platform for students",
        "Clean drinking water system",
        "Solar energy for rural homes"
    ],
    "actual_sdg": [
        "SDG 3",
        "SDG 4",
        "SDG 6",
        "SDG 7"
    ]
}

df = pd.DataFrame(data)
df

In [ ]:
evaluate(df)